# TrOCR Sequential Fine-Tuning - Kaggle (T4 x2)

Continues training a model you **already fine-tuned** (for example `TrOcr-50k-broken-samples`) on **new samples**,
instead of starting again from Microsoft's base model. To train from the base, use `trocr-finetuning-code.ipynb`.

**What this notebook does**
- Starts from your fine-tuned model, at half the first run's learning rate.
- Mixes old training samples in with the new ones (**replay**), so the model does not forget what it learned.
- Tests on the **same locked test set** as the model you start from (the old dataset's `test/`), so the scores compare.
- Scores three models on the same images: Microsoft's base, the model you start from, and the new model.
  It counts the test images the new model fixed and broke, and checks whether that difference could be chance.
- Never trains on a test image: exact copies of test images are removed from training and validation.

**Setup before running**
1. Settings -> Accelerator -> **GPU T4 x2**, and Settings -> Internet -> **On**.
2. Add three inputs (Add Input -> Your Datasets):
   - **the model to continue from**: zip the folder `ml/models/TrOcr-50k-broken-samples` (the model CRMS uses)
     and upload it as a private dataset,
   - **the old dataset** that model was trained on (its `test/` is the locked test),
   - **the new dataset** (`manifest.csv` + `train/ val/ test/`).
3. In cell 4, set `CRMS_MODEL_KEY` (the name you will upload the new model as). Leave the three paths blank:
   cell 4 lists what it found and fills them in when it can tell them apart.
4. Run it with **Save Version -> Save & Run All**, so it keeps running if your browser disconnects.
   Expect about 1.5-2 hours for 20k new + 20k old samples. Red `AssertionError: can only test a child process`
   lines during training are harmless (the first notebook shows them too).

**What you get** (Output): `trocr-sequential.zip`. It holds the model, `evaluation-report.json` (its scores on the
locked test), `base-evaluation-report.json` (Microsoft's base on the same test) and `sequential-summary.json`
(every number of this run). `sequential-test-errors.csv` lists the test images the start or new model read wrong.

**Upload the new model only if cell 15 says so.** Scores are read the way CRMS reads (at most 32 tokens per crop),
so they can differ slightly from the first notebook, which allowed 40.

In [ ]:
# 1. (Optional) ensure a recent transformers. Kaggle usually has it preinstalled.
# Uncomment if you hit version issues.
# !pip install -q -U transformers

print('[OK] cell ready')

In [ ]:
# 2. Imports and small helpers
import os, gc, json, math, time, random, shutil, hashlib, zipfile, warnings, logging
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

import numpy as np
import pandas as pd
import torch
import transformers
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import (TrOCRProcessor, VisionEncoderDecoderModel,
                          get_linear_schedule_with_warmup)

WEIGHT_FILES = ('model.safetensors', 'pytorch_model.bin')


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as source:
        while chunk := source.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def weights_file_in(folder):
    return next((name for name in WEIGHT_FILES if os.path.isfile(os.path.join(folder, name))), None)


def read_json(path):
    if not os.path.isfile(path):
        return None
    with open(path, encoding='utf-8') as source:
        return json.load(source)


def find_inputs(root):
    """Folders under `root` that hold a dataset (manifest.csv) or a model (config.json + weights)."""
    datasets, models = [], []
    for here, dirs, files in os.walk(root):
        dirs.sort()
        names = set(files)
        if 'manifest.csv' in names:
            datasets.append(os.path.normpath(here))
            dirs[:] = [d for d in dirs if d not in ('train', 'val', 'test')]  # do not walk the images
        if 'config.json' in names and names & set(WEIGHT_FILES):
            models.append(os.path.normpath(here))
    return datasets, models


print('[OK] Imports | torch', torch.__version__, '| transformers', transformers.__version__)

In [ ]:
# 3. GPU check
print('CUDA available :', torch.cuda.is_available())
n_gpu = torch.cuda.device_count()
print('GPU count      :', n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 4. CONFIG - inputs, names and training settings
MODEL_NAME = 'microsoft/trocr-base-handwritten'
MODEL_REVISION = 'eaacaf452b06415df8f10bb6fad3a4c11e609406'  # same pin as the first notebook and CRMS ml/models/base
KAGGLE_INPUT = '/kaggle/input'
WORKING_DIR = '/kaggle/working'

# Inputs. Leave them blank the first time: this cell lists what it found and fills in what it can tell apart.
START_MODEL_DIR = ''  # the fine-tuned model to continue from (a folder with config.json + model.safetensors)
OLD_DATA_ROOT = ''    # the dataset that model was trained on; its test/ is the locked test
NEW_DATA_ROOT = ''    # the new samples: manifest.csv + train/ val/ test/

# Names written into the CRMS reports.
CRMS_MODEL_KEY = ''     # the name you will upload the new model as, e.g. 'TrOcr-50k-plus-20k'
DATASET_LABEL = ''      # name of the OLD dataset (the locked test). Blank = the name in the start model's report
NEW_DATASET_LABEL = ''  # name of the new dataset. Blank = its folder name

# Replay: old training samples mixed in with the new ones, so the model keeps what it learned.
REPLAY_RATIO = 1.0      # old samples per new sample: 1.0 = as many old as new, 0 = none
# Validation picks the best epoch. It mixes old and new samples, so forgetting counts against an epoch.
VAL_OLD_SAMPLES = 2500
VAL_NEW_SAMPLES = 2500

# Gentler than the first run, whose 50k profile used DECODER_LR=2e-5 and ENCODER_LR=1e-5.
cfg = dict(EPOCHS=2, BATCH_SIZE=16, GRAD_ACCUM_STEPS=8,
           DECODER_LR=1e-5, ENCODER_LR=5e-6, WARMUP_RATIO=0.05,
           EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40)
EVAL_BATCH_SIZE = 32
EVAL_MAX_NEW_TOKENS = 32   # CRMS reads at most 32 tokens per crop (ml/api/main.py), so the scores match the app
NUM_WORKERS = 2
SEED = 20261102
CHECK_DUPLICATE_IMAGES = True  # remove exact copies of test images from training and validation

OUTPUT_DIR = os.path.join(WORKING_DIR, 'trocr-sequential')
seed_everything(SEED)

# ---- Find the inputs -------------------------------------------------------------------
found_datasets, found_models = find_inputs(KAGGLE_INPUT)
print('Datasets found (folders with manifest.csv):')
for path in found_datasets or ['(none)']:
    print('   ', path)
print('Models found (folders with config.json + weights):')
for path in found_models or ['(none)']:
    print('   ', path)

START_MODEL_DIR, OLD_DATA_ROOT, NEW_DATA_ROOT = (
    os.path.normpath(p) if p else '' for p in (START_MODEL_DIR, OLD_DATA_ROOT, NEW_DATA_ROOT))
if not START_MODEL_DIR:
    if len(found_models) != 1:
        raise RuntimeError('Set START_MODEL_DIR to the model folder to continue from (listed above).')
    START_MODEL_DIR = found_models[0]
start_weights_file = weights_file_in(START_MODEL_DIR)
if not start_weights_file:
    raise FileNotFoundError(f'No model.safetensors in START_MODEL_DIR: {START_MODEL_DIR}')

# Its report must belong to its weights, or its earlier scores mean nothing here.
start_weights_sha256 = sha256_file(os.path.join(START_MODEL_DIR, start_weights_file))
start_report = read_json(os.path.join(START_MODEL_DIR, 'evaluation-report.json'))
if start_report is not None and start_report.get('weights_sha256') != start_weights_sha256:
    print("WARNING: the start model's evaluation-report.json was made for other weights; it is ignored.")
    start_report = None

if not (OLD_DATA_ROOT and NEW_DATA_ROOT):
    # The start model's report names its test set by the SHA-256 of that dataset's manifest.csv.
    tested_on = (start_report or {}).get('manifest_sha256')
    if not OLD_DATA_ROOT and tested_on:
        matches = [p for p in found_datasets if p != NEW_DATA_ROOT
                   and sha256_file(os.path.join(p, 'manifest.csv')) == tested_on]
        if len(matches) == 1:
            OLD_DATA_ROOT = matches[0]
            print('OLD_DATA_ROOT picked: its manifest.csv is the one the start model was tested on.')
    rest = [p for p in found_datasets if p not in (OLD_DATA_ROOT, NEW_DATA_ROOT)]
    if OLD_DATA_ROOT and not NEW_DATA_ROOT and len(rest) == 1:
        NEW_DATA_ROOT = rest[0]
    elif NEW_DATA_ROOT and not OLD_DATA_ROOT and len(rest) == 1:
        OLD_DATA_ROOT = rest[0]
if not (OLD_DATA_ROOT and NEW_DATA_ROOT):
    raise RuntimeError('Set OLD_DATA_ROOT (the dataset the start model was trained on) and '
                       'NEW_DATA_ROOT (the new samples) to folders listed above.')
for name, root in (('OLD_DATA_ROOT', OLD_DATA_ROOT), ('NEW_DATA_ROOT', NEW_DATA_ROOT)):
    if not os.path.isfile(os.path.join(root, 'manifest.csv')):
        raise FileNotFoundError(f'No manifest.csv in {name}: {root}')
if OLD_DATA_ROOT == NEW_DATA_ROOT:
    raise RuntimeError('OLD_DATA_ROOT and NEW_DATA_ROOT are the same folder.')

parent_key = str((start_report or {}).get('model_key') or os.path.basename(START_MODEL_DIR)).strip()
CRMS_MODEL_KEY = CRMS_MODEL_KEY.strip() or os.path.basename(OUTPUT_DIR)
DATASET_LABEL = (DATASET_LABEL.strip()
                 or str((start_report or {}).get('dataset') or os.path.basename(OLD_DATA_ROOT)).strip())
NEW_DATASET_LABEL = NEW_DATASET_LABEL.strip() or os.path.basename(NEW_DATA_ROOT)

print()
print('START_MODEL_DIR =', START_MODEL_DIR, f'-> continuing from {parent_key!r}')
print('OLD_DATA_ROOT   =', OLD_DATA_ROOT, f'-> locked test, reported as {DATASET_LABEL!r}')
print('NEW_DATA_ROOT   =', NEW_DATA_ROOT, f'-> new samples, {NEW_DATASET_LABEL!r}')
print('New model name  =', repr(CRMS_MODEL_KEY))
print('Settings        :', cfg)
print(f'Replay ratio {REPLAY_RATIO} | seed {SEED} | reading limit {EVAL_MAX_NEW_TOKENS} tokens')

In [ ]:
# 5. Metrics (dependency-free CER / WER / exact match) and a chance check
def _levenshtein(ref, hyp):
    m, n = len(ref), len(hyp)
    if m == 0:
        return n
    if n == 0:
        return m
    prev = list(range(n + 1))
    for i in range(1, m + 1):
        curr = [i] + [0] * n
        for j in range(1, n + 1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            curr[j] = min(prev[j]+1, curr[j-1]+1, prev[j-1]+cost)
        prev = curr
    return prev[n]


def compute_metrics(references, hypotheses):
    tce = tc = twe = tw = exact = 0
    for ref, hyp in zip(references, hypotheses):
        ref, hyp = str(ref), str(hyp)
        tce += _levenshtein(list(ref), list(hyp)); tc += len(ref)
        rw, hw = ref.split(), hyp.split()
        twe += _levenshtein(rw, hw); tw += len(rw)
        if ref == hyp:
            exact += 1
    n = len(references)
    return {'cer': (tce/tc) if tc else 0.0, 'wer': (twe/tw) if tw else 0.0,
            'accuracy': (exact/n) if n else 0.0, 'exact': exact, 'total': n}


def print_metrics(m, title='EVALUATION METRICS'):
    print('='*56); print(title); print('='*56)
    print(f"  Samples evaluated : {m['total']}")
    print(f"  Exact match       : {m['exact']}/{m['total']} ({m['accuracy']*100:.2f}%)")
    print(f"  CER               : {m['cer']*100:.2f}%")
    print(f"  WER               : {m['wer']*100:.2f}%")
    print('='*56)


def mcnemar_p(fixed, broke):
    """Exact two-sided McNemar test on the images only one of two models read right.
    Below 0.05, the difference between the two models is unlikely to be chance."""
    n = fixed + broke
    if n == 0:
        return 1.0
    tail = sum(math.comb(n, i) for i in range(min(fixed, broke) + 1)) / 2 ** n
    return min(1.0, 2 * tail)


print('[OK] Metrics')

In [ ]:
# 6. The start model's processor, and the decoder-start fix from the first notebook
if start_report:
    m = start_report.get('metrics', {})
    print(f"Start model's own report: {start_report.get('dataset')} test, {start_report.get('sample_count')} images, "
          f"CER {float(m.get('cer', 0))*100:.2f}%  WER {float(m.get('wer', 0))*100:.2f}%  "
          f"exact {float(m.get('exact_match', 0))*100:.2f}%")
else:
    print('The start model has no matching evaluation-report.json; it is scored below either way.')

processor = TrOCRProcessor.from_pretrained(START_MODEL_DIR)
tokenizer = processor.tokenizer


def prepare_for_reading(model, proc):
    """TrOCR starts decoding from id 2 (sep/eos). Keep config and generation_config in step."""
    tok = proc.tokenizer
    model.config.decoder_start_token_id = tok.sep_token_id
    model.config.eos_token_id = tok.sep_token_id
    model.config.pad_token_id = tok.pad_token_id
    model.config.vocab_size = model.config.decoder.vocab_size
    model.generation_config.decoder_start_token_id = model.config.decoder_start_token_id
    model.generation_config.eos_token_id = model.config.eos_token_id
    model.generation_config.pad_token_id = model.config.pad_token_id
    model.generation_config.max_length = cfg['MAX_LABEL_LENGTH']
    return model


print('[OK] Processor loaded from the start model')

In [ ]:
# 7. DATA - new samples + replayed old samples for training; the old test stays locked
def read_manifest(root, origin):
    frame = pd.read_csv(os.path.join(root, 'manifest.csv'), keep_default_na=False)
    missing = {'filename', 'label', 'split'} - set(frame.columns)
    if missing:
        raise ValueError(f'{root}/manifest.csv has no {sorted(missing)} column')
    # The same filter as the first notebook, so the locked test is exactly its test set.
    frame = frame[frame['label'].astype(str).str.strip() != '']
    frame = frame[frame['label'].astype(str).str.upper() != 'UNREADABLE'].copy()
    frame['label'] = frame['label'].astype(str).str.strip()
    frame['path'] = [os.path.join(root, str(split), str(name))
                     for split, name in zip(frame['split'], frame['filename'])]
    frame['origin'] = origin
    return frame.reset_index(drop=True)


def split_of(frame, split):
    return frame[frame['split'] == split].reset_index(drop=True)


def md5_of(path):
    with open(path, 'rb') as source:
        return hashlib.md5(source.read()).hexdigest()


old_all, new_all = read_manifest(OLD_DATA_ROOT, 'old'), read_manifest(NEW_DATA_ROOT, 'new')
old_train, old_val, old_test = (split_of(old_all, s) for s in ('train', 'val', 'test'))
new_train, new_val, new_test = (split_of(new_all, s) for s in ('train', 'val', 'test'))
if old_test.empty:
    raise ValueError('The old dataset has no test split, so there is no locked test to compare on.')
if new_train.empty:
    raise ValueError('The new dataset has no train split.')

frames = {'old train': old_train, 'old val': old_val, 'old test': old_test,
          'new train': new_train, 'new val': new_val, 'new test': new_test}
missing_files = [p for frame in frames.values() for p in frame['path'] if not os.path.isfile(p)]
if missing_files:
    raise FileNotFoundError(f'{len(missing_files)} images named in the manifests are missing, '
                            f'e.g. {missing_files[:3]}')

# Exact copies (the same file bytes) of test images are never trained or validated on.
if CHECK_DUPLICATE_IMAGES:
    with ThreadPoolExecutor(max_workers=8) as pool:
        for name, frame in frames.items():
            frame['md5'] = list(tqdm(pool.map(md5_of, frame['path']), total=len(frame),
                                     desc=f'Checking {name} images', leave=False))
else:
    for frame in frames.values():
        frame['md5'] = frame['path']  # without the check, only the very same file counts as a copy

test_md5 = set(old_test['md5']) | set(new_test['md5'])
new_train_used = new_train[~new_train['md5'].isin(test_md5)].reset_index(drop=True)
replay_pool = old_train[~old_train['md5'].isin(test_md5)]
n_replay = min(len(replay_pool), int(round(len(new_train_used) * REPLAY_RATIO)))
replay = replay_pool.sample(n=n_replay, random_state=SEED).reset_index(drop=True)
train_rows = pd.concat([new_train_used, replay], ignore_index=True)
train_md5 = set(train_rows['md5'])


def pick_val(frame, limit):
    usable = frame[~frame['md5'].isin(test_md5 | train_md5)]
    return usable.sample(n=min(limit, len(usable)), random_state=SEED), len(frame) - len(usable)


val_old, val_old_removed = pick_val(old_val, VAL_OLD_SAMPLES)
val_new, val_new_removed = pick_val(new_val, VAL_NEW_SAMPLES)
val_df = pd.concat([val_old, val_new], ignore_index=True)
if val_df.empty:
    raise ValueError('No validation samples: the datasets need a val split.')

# Texts any of the models trained on (the start model trained on all of the old train split).
seen_texts = set(old_train['label']) | set(new_train_used['label'])
for frame in (old_test, new_test):
    frame['label_seen'] = frame['label'].isin(seen_texts)

# Label lengths in tokens: training cuts at MAX_LABEL_LENGTH, CRMS reads at most EVAL_MAX_NEW_TOKENS.
texts = sorted(set(pd.concat([train_rows['label'], val_df['label'], old_test['label'], new_test['label']])))
token_len = dict(zip(texts, (len(ids) for ids in tokenizer(texts)['input_ids'])))
train_cut = int(sum(token_len[t] > cfg['MAX_LABEL_LENGTH'] for t in train_rows['label']))
test_cut = {key: int(sum(token_len[t] > EVAL_MAX_NEW_TOKENS for t in frame['label']))
            for key, frame in (('old_test', old_test), ('new_test', new_test))}

data_summary = {
    'new_train_samples': len(new_train_used),
    'new_train_copies_of_test_removed': len(new_train) - len(new_train_used),
    'replay_samples': len(replay),
    'old_train_copies_of_test_not_replayed': len(old_train) - len(replay_pool),
    'training_samples': len(train_rows),
    'val_old_samples': len(val_old),
    'val_new_samples': len(val_new),
    'val_copies_removed': val_old_removed + val_new_removed,
    'locked_test_samples': len(old_test),
    'new_test_samples': len(new_test),
    'locked_test_texts_seen_in_training': int(old_test['label_seen'].sum()),
    'new_test_texts_seen_in_training': int(new_test['label_seen'].sum()),
    'duplicate_check': CHECK_DUPLICATE_IMAGES,
    'new_test_images_start_model_trained_on': (int(new_test['md5'].isin(set(old_train['md5'])).sum())
                                               if CHECK_DUPLICATE_IMAGES else None),
    'images_in_both_tests': (int(new_test['md5'].isin(set(old_test['md5'])).sum())
                             if CHECK_DUPLICATE_IMAGES else None),
    'longest_label_tokens': max(token_len.values()),
    'training_labels_cut_at_max_label_length': train_cut,
    'test_labels_longer_than_crms_reads': test_cut,
}

print(f"Training    : {len(new_train_used):,} new + {len(replay):,} old (replay) = {len(train_rows):,} samples")
if data_summary['new_train_copies_of_test_removed']:
    print(f"              exact copies of test images left out: {data_summary['new_train_copies_of_test_removed']:,}")
print(f"Validation  : {len(val_old):,} old + {len(val_new):,} new")
if data_summary['val_copies_removed']:
    print(f"              copies of test or training images left out: {data_summary['val_copies_removed']:,}")
print(f"Locked test : {len(old_test):,} images from {DATASET_LABEL} | "
      f"texts the models trained on: {data_summary['locked_test_texts_seen_in_training']:,}")
print(f"New test    : {len(new_test):,} images from {NEW_DATASET_LABEL} | "
      f"texts the models trained on: {data_summary['new_test_texts_seen_in_training']:,}")
if val_new.empty:
    print('NOTE: the new dataset has no usable val split, so the best epoch is picked on old samples only.')
if data_summary['new_test_images_start_model_trained_on']:
    print('NOTE: new test images the start model trained on: '
          f"{data_summary['new_test_images_start_model_trained_on']:,} (its new-test score is too high by those)")
print(f"Longest label: {data_summary['longest_label_tokens']} tokens")
if train_cut:
    print(f"WARNING: training labels longer than MAX_LABEL_LENGTH ({cfg['MAX_LABEL_LENGTH']} tokens), "
          f'cut in training: {train_cut:,}')
for key, count in test_cut.items():
    if count:
        print(f'WARNING: {key} labels longer than CRMS reads ({EVAL_MAX_NEW_TOKENS} tokens): {count:,}. '
              'They count as wrong here, as they would in CRMS.')

In [ ]:
# 8. Dataset and loader
class HandwrittenDataset(Dataset):
    def __init__(self, frame, processor, max_len):
        self.paths = frame['path'].tolist()
        self.labels = frame['label'].tolist()
        self.processor = processor
        self.max_len = max_len

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as opened:
            image = opened.convert('RGB')
        pixel_values = self.processor(images=image, return_tensors='pt').pixel_values.squeeze(0)
        labels = self.processor.tokenizer(self.labels[idx], padding='max_length', max_length=self.max_len,
                                          truncation=True, return_tensors='pt').input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}


train_ds = HandwrittenDataset(train_rows, processor, cfg['MAX_LABEL_LENGTH'])
train_loader = DataLoader(train_ds, batch_size=cfg['BATCH_SIZE'], shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=True,
                          generator=torch.Generator().manual_seed(SEED))
if len(train_loader) == 0:
    raise ValueError('Fewer training samples than one batch.')
print('Training batches per epoch:', len(train_loader))

In [ ]:
# 9. Reading and scoring - the way CRMS reads: greedy, at most EVAL_MAX_NEW_TOKENS tokens
@torch.no_grad()
def read_images(paths, model, proc, desc):
    model.eval()
    texts = []
    for start in tqdm(range(0, len(paths), EVAL_BATCH_SIZE), desc=desc, leave=False):
        images = []
        for path in paths[start:start + EVAL_BATCH_SIZE]:
            with Image.open(path) as opened:
                images.append(opened.convert('RGB'))
        pixel_values = proc(images=images, return_tensors='pt').pixel_values.to(device)
        generated = model.generate(pixel_values, max_new_tokens=EVAL_MAX_NEW_TOKENS, use_cache=True)
        texts.extend(text.strip() for text in proc.batch_decode(generated, skip_special_tokens=True))
    return texts


def score(frame, model, proc, title=None):
    """Metrics plus each image's reading; None for an empty set."""
    if frame is None or len(frame) == 0:
        return None
    predictions = read_images(frame['path'].tolist(), model, proc, title or 'Reading')
    result = compute_metrics(frame['label'].tolist(), predictions)
    result['predictions'] = predictions
    result['correct'] = [p == r for p, r in zip(predictions, frame['label'])]
    if title:
        print_metrics(result, title)
    return result


def score_subset(frame, result, mask):
    refs = [r for r, keep in zip(frame['label'], mask) if keep]
    hyps = [h for h, keep in zip(result['predictions'], mask) if keep]
    return compute_metrics(refs, hyps)


def public(result):
    """Only the numbers, for the reports."""
    if result is None:
        return None
    return {key: result[key] for key in ('cer', 'wer', 'accuracy', 'exact', 'total')}


print('[OK] Evaluator')

In [ ]:
# 10. BEFORE (1 of 2): Microsoft's base model on the same tests - for base-evaluation-report.json
results = {'base': {}, 'start': {}, 'new': {}}
base_processor = TrOCRProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
base_model = prepare_for_reading(
    VisionEncoderDecoderModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION), base_processor).to(device)
results['base']['old_test'] = score(old_test, base_model, base_processor, 'BASE MODEL - LOCKED TEST (old dataset)')
results['base']['new_test'] = score(new_test, base_model, base_processor, 'BASE MODEL - NEW TEST')
# The pinned weights scored above. CRMS checks this hash against its own ml/models/base.
base_weights_sha256 = sha256_file(hf_hub_download(MODEL_NAME, 'model.safetensors', revision=MODEL_REVISION))
del base_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 11. BEFORE (2 of 2): the model you continue from, on the same tests and on the validation mix
raw_model = prepare_for_reading(VisionEncoderDecoderModel.from_pretrained(START_MODEL_DIR), processor).to(device)
print('Params:', f'{sum(p.numel() for p in raw_model.parameters()):,}')
results['start']['old_test'] = score(old_test, raw_model, processor, f'START MODEL ({parent_key}) - LOCKED TEST')
results['start']['new_test'] = score(new_test, raw_model, processor, f'START MODEL ({parent_key}) - NEW TEST')
start_val = score(val_df, raw_model, processor)

if start_report:
    said = float(start_report.get('metrics', {}).get('exact_match', 0))
    measured = results['start']['old_test']
    print(f"Its own report said exact {said*100:.2f}% on {start_report.get('sample_count')} images; "
          f"measured here: {measured['accuracy']*100:.2f}% on {measured['total']}.")
    if int(start_report.get('sample_count', -1)) != measured['total'] or abs(said - measured['accuracy']) > 0.005:
        print('WARNING: not the same result, so OLD_DATA_ROOT may not be the test set its report used.')

In [ ]:
# 12. Optimizer (separate encoder/decoder learning rates), warmup + linear decay, both GPUs
encoder_params = list(raw_model.encoder.parameters())
decoder_params = [p for n, p in raw_model.named_parameters() if not n.startswith('encoder.')]
optimizer = AdamW([
    {'params': decoder_params, 'lr': cfg['DECODER_LR']},
    {'params': encoder_params, 'lr': cfg['ENCODER_LR']},
])
updates_per_epoch = math.ceil(len(train_loader) / cfg['GRAD_ACCUM_STEPS'])
total_updates = updates_per_epoch * cfg['EPOCHS']
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(total_updates * cfg['WARMUP_RATIO']), total_updates)

# Both T4s for the training forward pass. Reading stays on raw_model (1 GPU).
model = raw_model
if n_gpu > 1:
    model = torch.nn.DataParallel(raw_model)
    print(f'Using DataParallel across {n_gpu} GPUs')
scaler = GradScaler('cuda')
print(f'[OK] Optimizer | {updates_per_epoch} updates per epoch, {total_updates} in all')

In [ ]:
# 13. TRAINING - mixed precision, gradient accumulation, best epoch by validation CER, early stop
if os.path.isdir(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)  # a re-run must never package files from an earlier run
os.makedirs(OUTPUT_DIR)
val_old_mask = (val_df['origin'] == 'old').tolist()
val_new_mask = [not is_old for is_old in val_old_mask]


def val_summary(result):
    summary = {'val_cer': result['cer'], 'val_exact': result['accuracy']}
    for name, mask in (('old', val_old_mask), ('new', val_new_mask)):
        summary[f'val_cer_{name}'] = score_subset(val_df, result, mask)['cer'] if any(mask) else None
    return summary


def describe(summary):
    text = f"val CER {summary['val_cer']*100:.2f}%"
    for name in ('old', 'new'):
        if summary[f'val_cer_{name}'] is not None:
            text += f" | {name} samples {summary[f'val_cer_{name}']*100:.2f}%"
    return text


epoch_log = [{'epoch': 0, 'train_loss': None, **val_summary(start_val)}]
print(f'  Epoch 0 (the start model): {describe(epoch_log[0])}')
best_cer, best_epoch, patience = float('inf'), None, 0
started = time.time()

for epoch in range(1, cfg['EPOCHS'] + 1):
    model.train()
    running = 0.0
    optimizer.zero_grad(set_to_none=True)
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{cfg["EPOCHS"]}')
    for step, batch in enumerate(progress, start=1):
        pixel_values = batch['pixel_values'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        with autocast('cuda'):
            loss = model(pixel_values=pixel_values, labels=labels).loss.mean() / cfg['GRAD_ACCUM_STEPS']
        scaler.scale(loss).backward()
        running += loss.item() * cfg['GRAD_ACCUM_STEPS']
        # Step every GRAD_ACCUM_STEPS batches, and on the last batch so no gradient is thrown away.
        if step % cfg['GRAD_ACCUM_STEPS'] == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)  # unscale before clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
        progress.set_postfix(loss=f'{running/step:.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')

    val = score(val_df, raw_model, processor)
    summary = {'epoch': epoch, 'train_loss': running / len(train_loader), **val_summary(val)}
    epoch_log.append(summary)
    print(f"  Epoch {epoch}: train_loss={summary['train_loss']:.4f}  {describe(summary)}")
    if val['cer'] < best_cer:
        best_cer, best_epoch, patience = val['cer'], epoch, 0
        raw_model.save_pretrained(OUTPUT_DIR)
        processor.save_pretrained(OUTPUT_DIR)
        print('    saved as the best epoch so far')
    else:
        patience += 1
        print(f"    no improvement ({patience}/{cfg['EARLY_STOP_PATIENCE']})")
        if patience >= cfg['EARLY_STOP_PATIENCE']:
            print('    early stopping')
            break

training_minutes = (time.time() - started) / 60
print(f'\nDone in {training_minutes:.1f} min. Best epoch: {best_epoch} (val CER {best_cer*100:.2f}%); '
      f"the start model had {start_val['cer']*100:.2f}%.")

In [ ]:
# 14. AFTER: the best epoch, on the same tests
for name in ('model', 'optimizer', 'scheduler', 'scaler', 'raw_model', 'encoder_params', 'decoder_params',
             'batch', 'pixel_values', 'labels', 'loss'):
    globals().pop(name, None)  # drop every reference to the training model, to free the GPU
gc.collect()
torch.cuda.empty_cache()
if best_epoch is None:
    raise RuntimeError('No epoch was trained, so there is no new model.')
best_model = prepare_for_reading(VisionEncoderDecoderModel.from_pretrained(OUTPUT_DIR), processor).to(device)
results['new']['old_test'] = score(old_test, best_model, processor, f'NEW MODEL ({CRMS_MODEL_KEY}) - LOCKED TEST')
results['new']['new_test'] = score(new_test, best_model, processor, f'NEW MODEL ({CRMS_MODEL_KEY}) - NEW TEST')

In [ ]:
# 15. COMPARISON - the same images for all three models, and what the new model changed
def pct(value):
    return '-' if value is None else f'{value*100:.2f}%'


tests = [('old_test', f'LOCKED TEST - {DATASET_LABEL} ({len(old_test):,} images)', old_test)]
if len(new_test):
    tests.append(('new_test', f'NEW TEST - {NEW_DATASET_LABEL} ({len(new_test):,} images)', new_test))
widths = (14, 18, 18)
changes, unseen_scores = {}, {}
for key, title, frame in tests:
    trio = [results[name][key] for name in ('base', 'start', 'new')]
    print('=' * 70)
    print(title)
    print('-' * 70)
    print(f"{'':16}{'Base':>14}{'Start model':>18}{'New model':>18}")
    for metric, label in (('cer', 'CER'), ('wer', 'WER'), ('accuracy', 'Exact match')):
        print(f'{label:16}' + ''.join(f'{pct(r[metric]):>{w}}' for r, w in zip(trio, widths)))
    unseen = [not seen for seen in frame['label_seen']]
    if any(unseen):
        exact_unseen = [score_subset(frame, r, unseen)['accuracy'] for r in trio]
        unseen_scores[key] = {'images': sum(unseen), **dict(zip(('base', 'start', 'new'), exact_unseen))}
        print(f"{'Unseen texts':16}" + ''.join(f'{pct(v):>{w}}' for v, w in zip(exact_unseen, widths))
              + f'   exact match on the {sum(unseen):,} images whose text no model trained on')
    start, new = trio[1], trio[2]
    fixed = sum(1 for s, n in zip(start['correct'], new['correct']) if n and not s)
    broke = sum(1 for s, n in zip(start['correct'], new['correct']) if s and not n)
    p = mcnemar_p(fixed, broke)
    changes[key] = {'fixed': fixed, 'broke': broke, 'mcnemar_p': p}
    print(f'New model vs start model: fixed {fixed:,} images, broke {broke:,}. '
          f'Chance check p = {p:.3f} ({"unlikely to be chance" if p < 0.05 else "could be chance"})')
print('=' * 70)


def direction(change):
    if change['mcnemar_p'] >= 0.05:
        return 'same'
    return 'better' if change['fixed'] > change['broke'] else 'worse'


WORDS = {'better': 'better', 'worse': 'WORSE', 'same': 'about the same (any difference could be chance)'}
old_way = direction(changes['old_test'])
new_way = direction(changes['new_test']) if 'new_test' in changes else None
verdict = [f'Locked test (old samples): {WORDS[old_way]}.']
if new_way:
    verdict.append(f'New test (new samples): {WORDS[new_way]}.')
if old_way == 'worse':
    verdict.append('DO NOT UPLOAD: it got worse on the old samples (it forgot). '
                   'Run again with a higher REPLAY_RATIO or lower learning rates.')
elif new_way == 'worse':
    verdict.append('DO NOT UPLOAD: it got worse on the new samples.')
elif 'better' in (old_way, new_way):
    where = ' and '.join(name for name, way in (('the old samples', old_way), ('the new samples', new_way))
                         if way == 'better')
    verdict.append(f'UPLOAD {CRMS_MODEL_KEY!r}: better on {where}, and worse on none. '
                   f'Keep {parent_key!r} installed so you can switch back.')
else:
    verdict.append(f'NO CLEAR GAIN: it reads about as well as {parent_key!r}. Keep using {parent_key!r}; '
                   'this run still counts as a measured experiment.')
print('VERDICT')
for line in verdict:
    print('  ' + line)

In [ ]:
# 16. Reports - CRMS scores on the locked test, plus a summary of everything in this run
weights_file = weights_file_in(OUTPUT_DIR)
if weights_file is None:
    raise FileNotFoundError('The best epoch was not saved.')
manifest_sha256 = sha256_file(os.path.join(OLD_DATA_ROOT, 'manifest.csv'))
evaluated_at = datetime.now(timezone.utc).isoformat()


def crms_report(model_key, metrics, weights_name, weights_sha256):
    return {
        'schema_version': 1,
        'model_key': model_key,
        'base_model': MODEL_NAME,
        'base_model_revision': MODEL_REVISION,
        'dataset': DATASET_LABEL,
        'manifest_sha256': manifest_sha256,
        'split': 'test',
        'sample_count': int(metrics['total']),
        'metrics': {
            'cer': float(metrics['cer']),
            'wer': float(metrics['wer']),
            'exact_match': float(metrics['accuracy']),
        },
        'evaluated_at': evaluated_at,
        'weights_file': weights_name,
        'weights_sha256': weights_sha256,
    }


# Both reports are on the locked test, so CRMS sees one test set and compares base and new fairly.
report = crms_report(CRMS_MODEL_KEY, results['new']['old_test'], weights_file,
                     sha256_file(os.path.join(OUTPUT_DIR, weights_file)))
report['parent_model'] = {'key': parent_key, 'weights_sha256': start_weights_sha256}
report['training'] = {
    'kind': 'sequential', 'new_dataset': NEW_DATASET_LABEL,
    'new_train_samples': data_summary['new_train_samples'], 'replay_samples': data_summary['replay_samples'],
    'epochs_run': len(epoch_log) - 1, 'best_epoch': best_epoch, 'hyperparameters': cfg, 'seed': SEED,
}
if results['new'].get('new_test'):
    report['extra_test_sets'] = {NEW_DATASET_LABEL: public(results['new']['new_test'])}
base_report = crms_report('base', results['base']['old_test'], 'model.safetensors', base_weights_sha256)

summary = {
    'schema': 'crms-sequential-summary/1',
    'created_at': evaluated_at,
    'start_model': {'key': parent_key, 'dir': START_MODEL_DIR, 'weights_sha256': start_weights_sha256,
                    'its_report': start_report},
    'new_model': {'key': CRMS_MODEL_KEY, 'weights_sha256': report['weights_sha256'], 'best_epoch': best_epoch},
    'base_model': {'name': MODEL_NAME, 'revision': MODEL_REVISION, 'weights_sha256': base_weights_sha256},
    'datasets': {'old': {'label': DATASET_LABEL, 'root': OLD_DATA_ROOT, 'manifest_sha256': manifest_sha256},
                 'new': {'label': NEW_DATASET_LABEL, 'root': NEW_DATA_ROOT,
                         'manifest_sha256': sha256_file(os.path.join(NEW_DATA_ROOT, 'manifest.csv'))}},
    'data': data_summary,
    'settings': {**cfg, 'REPLAY_RATIO': REPLAY_RATIO, 'VAL_OLD_SAMPLES': VAL_OLD_SAMPLES,
                 'VAL_NEW_SAMPLES': VAL_NEW_SAMPLES, 'EVAL_MAX_NEW_TOKENS': EVAL_MAX_NEW_TOKENS, 'SEED': SEED},
    'scores': {model_name: {test: public(r) for test, r in by_test.items()}
               for model_name, by_test in results.items()},
    'unseen_texts_exact_match': unseen_scores,
    'new_vs_start': changes,
    'epoch_log': epoch_log,
    'verdict': verdict,
    'training_minutes': round(training_minutes, 1),
    'versions': {'torch': torch.__version__, 'transformers': transformers.__version__},
}

for name, content in (('evaluation-report.json', report), ('base-evaluation-report.json', base_report),
                      ('sequential-summary.json', summary)):
    with open(os.path.join(OUTPUT_DIR, name), 'w', encoding='utf-8') as output:
        json.dump(content, output, indent=2)
shutil.copy2(os.path.join(OUTPUT_DIR, 'sequential-summary.json'), os.path.join(WORKING_DIR, 'sequential-summary.json'))

# Every test image the start or new model read wrong, with all three readings.
rows = []
for key, _, frame in tests:
    base, start, new = (results[name][key] for name in ('base', 'start', 'new'))
    for i in range(len(frame)):
        if not (start['correct'][i] and new['correct'][i]):
            rows.append({'test_set': key, 'filename': frame['filename'][i], 'label': frame['label'][i],
                         'base': base['predictions'][i], 'start_model': start['predictions'][i],
                         'new_model': new['predictions'][i], 'text_seen_in_training': bool(frame['label_seen'][i])})
errors_csv = os.path.join(WORKING_DIR, 'sequential-test-errors.csv')
pd.DataFrame(rows, columns=['test_set', 'filename', 'label', 'base', 'start_model', 'new_model',
                            'text_seen_in_training']).to_csv(errors_csv, index=False)

print('CRMS report (new model, locked test):')
print(json.dumps(report, indent=2))
print('Base report: exact', f"{base_report['metrics']['exact_match']*100:.2f}%", 'on the same', base_report['sample_count'], 'images')
print('Errors list:', errors_csv, f'({len(rows):,} rows)')

In [ ]:
# 17. Package the new model and its reports for CRMS
archive = shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
with zipfile.ZipFile(archive) as packed_zip:
    packed = set(packed_zip.namelist())
for required in ('config.json', weights_file, 'evaluation-report.json', 'base-evaluation-report.json',
                 'sequential-summary.json'):
    assert required in packed, f'{required} is missing from the ZIP - rerun cells 13 to 16.'
print('zipped ->', archive)
print(f'Upload it in the CRMS OCR Workspace as {CRMS_MODEL_KEY!r} only if cell 15 says UPLOAD.')
print(f'Keep {parent_key!r} installed so you can switch back.')